# Early Warning Decision Engine: Exploration, Temporal Audit, and Data Leakage Benchmark
**دانشجو:** شایان گنجی  
**آزمایشگاه:** فناوری های مالی داتین- ترم سوم  
**پایگاه‌داده هدف:** PostgreSQL 16.4 | دیتابیس `olist_db`

In [1]:
import logging
import warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

from early_warning.config import DEFAULT_CONFIG
from early_warning.data.external_api import BrasilAPIHolidayClient, OpenMeteoWeatherBenchmarkClient
from early_warning.features.temporal_audit import get_audit_dataframe, ColumnLifecycleStatus
from early_warning.models.leakage_benchmark import run_leakage_benchmark

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment and early_warning package successfully initialized.')

2026/11/20 01:37:17 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at C:\Users\USER\anaconda3\envs\tf-env\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Environment and early_warning package successfully initialized.


## 1. Problem Formulation & Class Distribution Analysis
<div dir='rtl'>فرمول‌بندی مسئله در لحظه تایید سفارش و بررسی نرخ پایه نظرات منفی مشتریان</div>

In [2]:
engine = create_engine(DEFAULT_CONFIG.db_url)
reviews_query = '''
SELECT 
    review_score,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM olist.order_reviews
GROUP BY review_score
ORDER BY review_score;
'''
df_reviews = pd.read_sql_query(reviews_query, engine)
print(df_reviews.to_string(index=False))

# Plot score distribution
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(df_reviews['review_score'], df_reviews['percentage'], color=['firebrick', 'salmon', 'lightgray', 'cornflowerblue', 'steelblue'])
ax.set_xlabel('Review Score (1 = Worst, 5 = Best)')
ax.set_ylabel('Percentage of Total Orders (%)')
ax.set_title('Olist Customer Review Score Distribution')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

negative_base_rate = df_reviews[df_reviews['review_score'].isin([1, 2])]['percentage'].sum()
print(f'Negative Review Target Class Base Rate (Score 1 or 2): {negative_base_rate:.2f}%')

 review_score  order_count  percentage
            1        11424       11.51
            2         3151        3.18
            3         8179        8.24
            4        19142       19.29
            5        57328       57.78
Negative Review Target Class Base Rate (Score 1 or 2): 14.69%


## 2. Complete Schema Temporal Audit Registry
<div dir='rtl'>ممیزی چرخه حیات زمانی تمامی ستون‌های پایگاه‌داده نسبت به لحظه تصمیم‌گیری T0</div>

In [3]:
audit_df = get_audit_dataframe()
status_counts = audit_df['status'].value_counts()
print('Lifecycle Classification Summary:')
for status, count in status_counts.items():
    print(f'  - {status}: {count} columns')

# Display audit records for orders and reviews
audit_df[audit_df['table_name'].isin(['olist.orders', 'olist.order_reviews'])][['table_name', 'column_name', 'status', 'lifecycle_event', 'rationale']]

Lifecycle Classification Summary:
  - ALLOWED_AT_T0: 27 columns
  - METADATA_IDENTIFIER: 9 columns
  - FORBIDDEN_POST_T0: 7 columns
  - TARGET_ONLY: 2 columns


,table_name,column_name,status,lifecycle_event,rationale
0,olist.orders,order_id,METADATA_IDENTIFIER,Checkout,Primary key identifier of the order. Permissib...
1,olist.orders,customer_id,METADATA_IDENTIFIER,Checkout,Order-specific customer token. Permissible for...
2,olist.orders,order_status,FORBIDDEN_POST_T0,Order Lifecycle,"Terminal status ('delivered', 'canceled', 'shi..."
3,olist.orders,order_purchase_timestamp,ALLOWED_AT_T0,Checkout,Exact moment customer pressed purchase button....
4,olist.orders,order_approved_at,ALLOWED_AT_T0,Order Approval (T0),The definitive decision boundary T0. Order pay...
5,olist.orders,order_delivered_carrier_date,FORBIDDEN_POST_T0,Carrier Hand-off,Occurs 1 to 5 days after T0 when the seller dr...
6,olist.orders,order_delivered_customer_date,FORBIDDEN_POST_T0,Customer Delivery,Occurs 5 to 30 days after T0. Prime source of ...
7,olist.orders,order_estimated_delivery_date,ALLOWED_AT_T0,Checkout,Initial promise date given to the customer at ...
20,olist.order_reviews,review_id,TARGET_ONLY,Post-Delivery Survey,Unique survey response ID. Forbidden as a feat...
21,olist.order_reviews,order_id,METADATA_IDENTIFIER,Post-Delivery Survey,Foreign key referencing olist.orders.


## 3. External API Evaluation & Rejection Rationale
<div dir='rtl'>ارزیابی تجربی منابع داده بیرونی، اثبات عدم نشت داده تقویم رسمی و دلایل رد سایر منابع</div>

In [4]:
# Benchmark BrasilAPI Statutory Holiday Client
holiday_client = BrasilAPIHolidayClient()
holidays = holiday_client.fetch_holidays(years=(2016, 2017, 2018))
print(f'Successfully loaded {len(holidays)} Brazilian statutory holidays from local persistent cache.')
print('Sample statutory holidays:')
for h in holidays[:5]:
    print(f'  {h["date"]}: {h["name"]}')

# Benchmark Open-Meteo Weather Sample Client
weather_client = OpenMeteoWeatherBenchmarkClient()
weather_data = weather_client.fetch_weather_sample()
print(f'Loaded {len(weather_data.get("daily", {}).get("time", []))} daily weather observations for São Paulo.')

INFO:early_warning.data.external_api:Loaded 39 Brazilian holidays from local cache: D:\Lab\term 3\stage-4-shayan-ganji\data\external\holidays_br.json


Successfully loaded 39 Brazilian statutory holidays from local persistent cache.
Sample statutory holidays:
  2016-01-01: Confraternização mundial
  2016-02-08: Carnaval
  2016-02-09: Carnaval
  2016-03-25: Sexta-feira Santa
  2016-03-27: Páscoa
Loaded 31 daily weather observations for São Paulo.


## 4. Empirical Data Leakage Benchmark (Deceptive Gap Quantification)
<div dir='rtl'>آزمون تجربی مقایسه مدل سالم در برابر مدل آلوده به نشت و سنجش شکاف فریبنده عملکرد</div>

In [5]:
benchmark_result = run_leakage_benchmark()
print(benchmark_result.comparison_table.to_string(index=False))

print('\n' + '=' * 80)
print('ARCHITECTURAL TAKEAWAY ON DATA LEAKAGE:')
print('=' * 80)
print(benchmark_result.deceptive_narrative)

INFO:early_warning.features.sql_features:Loading feature dataset from PostgreSQL table 'olist.mv_order_early_warning_features'...


INFO:early_warning.features.sql_features:Loaded 99,281 records from database.


INFO:early_warning.features.sql_features:Enriching features with BrasilAPI statutory holiday calendar...


INFO:early_warning.data.external_api:Loaded 39 Brazilian holidays from local cache: D:\Lab\term 3\stage-4-shayan-ganji\data\external\holidays_br.json


INFO:early_warning.features.sql_features:Prepared feature matrix X shape: (98518, 21), target y base rate: 0.1467 (14,454 negative reviews out of 98,518)


INFO:early_warning.models.leakage_benchmark:Extracting post-T0 leaky features for benchmark comparison...


INFO:early_warning.models.leakage_benchmark:Training Clean Model on 58,759 orders with 21 features...


  File "C:\Users\USER\anaconda3\envs\tf-env\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\USER\anaconda3\envs\tf-env\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\USER\anaconda3\envs\tf-env\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\USER\anaconda3\envs\tf-env\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


INFO:early_warning.models.leakage_benchmark:Training Leaky Model on 58,759 orders with 25 features...


                    Metric Clean Model (T0 Compliant) Leaky Model (Contaminated) Deceptive Gap                                          Operational Impact in Live Production
  Capture Rate @ 10% Quota                     21.20%                     45.51%       +24.31% Clean captures legitimate pre-transit signals; Leaky crashes or predicts zero.
     Precision @ 10% Quota                     28.77%                     61.77%       +33.00%   Support team follows up with valid at-risk buyers vs. hallucinated accuracy.
    Operational Lift @ 10%                      2.12x                      4.55x        +2.43x                    True 2.1x operational efficiency over blind random calling.
                   ROC-AUC                     0.5928                     0.7753       +0.1825        Artificially inflated ranking metric caused by future transit duration.
PR-AUC (Average Precision)                     0.2320                     0.5152       +0.2832          Naively suggests 50% avera

In [6]:
# Visualize Deceptive Gap in Key Metrics
metrics_names = ['Capture Rate @ 10%', 'Precision @ 10%', 'ROC-AUC', 'PR-AUC']
clean_vals = [
    benchmark_result.clean_metrics['capture_rate_10pct'] * 100,
    benchmark_result.clean_metrics['precision_10pct'] * 100,
    benchmark_result.clean_metrics['roc_auc'] * 100,
    benchmark_result.clean_metrics['pr_auc'] * 100,
]
leaky_vals = [
    benchmark_result.leaky_metrics['capture_rate_10pct'] * 100,
    benchmark_result.leaky_metrics['precision_10pct'] * 100,
    benchmark_result.leaky_metrics['roc_auc'] * 100,
    benchmark_result.leaky_metrics['pr_auc'] * 100,
]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, clean_vals, width, label='Clean Model (T0 Compliant)', color='steelblue')
rects2 = ax.bar(x + width/2, leaky_vals, width, label='Leaky Model (Contaminated)', color='coral')

ax.set_ylabel('Score / Percentage (%)')
ax.set_title('Empirical Comparison: Clean vs. Leaky Model Across Key Metrics')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.grid(True, alpha=0.3)

for bar in rects1:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=9)
for bar in rects2:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Summary & Hand-off to Production System
<div dir='rtl'>جمع‌بندی یافته‌های اکتشافی و انتقال مدل آموزش‌دیده به نوت‌بوک سیستم عملیاتی</div>

Exploration and audit establish that:
1. **Decision Moment T0** is strictly bound to `order_approved_at`.
2. **Data Leakage** creates a deceptive +24.3% surge in Capture Rate @ 10% that would catastrophically fail in live deployment.
3. **External API**: BrasilAPI statutory calendar provides 100% leak-free predictive lift, while weather and inflation APIs are rejected.
4. **Next**: Proceed to `02_system.ipynb` for the production training, MLflow tracking, and capacity-constrained decision engine.